# The recommender on a season it has never seen — 2026

Notebook 14 tested the recommender on the second half of 2025: **91.3% top-3** against the naive
rule's 77.3%. That test is spent. This one is harder, and it is the last untouched data in the
project: **the 2026 season, January to June** — after an off-season of roster changes and
patches, with **agents that did not exist in 2025** and **a map that was not played in 2025**.

Nothing about 2026's line-ups has been looked at before this notebook. Its structure was
checked (row counts, dates, missing values), and its dates were read from vlr.gg by
`fetch_vlr_dates.py`, which was first checked against 2025 and matched on all 504 matches.

**This notebook was committed to git before it was run**, like notebooks 12 and 14.

## Decided in advance

1. **Methods and settings: exactly notebook 14's**, fixed at the end of notebook 13 and not
   changed since.

   | | Method | Recency half-life |
   |---|---|---|
   | **Headline** | **Gradient boosting** (`HistGradientBoostingClassifier`, defaults, `random_state=0`) | **45 days** |
   | | Classifier + map × agent columns | 21 days |
   | | Co-occurrence | 21 days |
   | | Classifier, plain | 45 days |
   | **The bar** | **Popularity — the naive rule** | 7 days |

2. **History.** All of 2024 and all of 2025. Then walk-forward: each week of 2026 is
   recommended using only line-ups played before that week, as in notebook 14.
3. **Measures**, as before: *top-1* and *top-3* (hide one agent, give the map and the other
   four, see where the hidden agent ranks; five questions per line-up), and *from scratch*
   (how many of the pros' five it picks from the map alone).
4. **Margin of error.** 5,000 resamples of whole matches, random seed 0. 95% ranges for the
   headline's top-1 and top-3, and for its **top-3 lead over the naive rule**.
5. **What counts as what:**
   - lead over the naive rule on top-3 with the bottom of its 95% range above zero →
     **beats the naive rule**;
   - a lead whose range includes zero → **ahead, but not distinguishable from the naive rule**;
   - no lead → reported plainly.
6. **Two breakdowns, and only these two**, for the headline method, both about what is new:
   - **questions whose hidden agent was never played in 2024 or 2025.** No method can
     recommend an agent before the pros have played it, so these start as guaranteed misses
     and can only be learned as 2026 goes on. How fast it catches up is the question;
   - **line-ups on maps not played at all in 2025** — returning from 2024, so with only old
     history behind them.
   No other slices.
7. **Expected: lower than notebook 14.** A new season resets much of the meta, and new
   agents are unrecommendable until they appear. A drop is not a failure; how big it is,
   and whether the lead over the naive rule survives, is the finding.
8. **The 2026 win results stay sealed.** This notebook reads line-ups only. It does not look
   at who won any 2026 map — that is reserved for the win models' own 2026 test.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier

sys.path.insert(0, str(Path.cwd().parent / "src"))

pd.set_option("display.width", 200)

from dataset import build_map_table
from recommend import Classifier, CoOccurrence, Popularity, lineup_table, summarise, walk_forward

SEED, RESAMPLES = 0, 5000
HEADLINE, BAR = "gradient boosting", "popularity (naive rule)"
METHODS = {
    "gradient boosting": lambda: Classifier(45, estimator=HistGradientBoostingClassifier(random_state=0)),
    "classifier + map x agent": lambda: Classifier(21, interactions=True),
    "co-occurrence": lambda: CoOccurrence(21),
    "classifier": lambda: Classifier(45),
    "popularity (naive rule)": lambda: Popularity(7),
}

## The line-ups

In [ ]:
history = pd.concat([lineup_table(build_map_table("vct_2024")), lineup_table(build_map_table())],
                    ignore_index=True)
to_score = lineup_table(build_map_table("vct_2026"))      # line-ups only -- no results are loaded

print(f"history : {len(history):,} line-ups, {history.played_at.min():%d %b %Y} to {history.played_at.max():%d %b %Y}")
print(f"scored  : {len(to_score):,} line-ups from {to_score.match_id.nunique()} matches, "
      f"{to_score.played_at.min():%d %b} to {to_score.played_at.max():%d %b %Y}, "
      f"over {to_score.played_at.dt.to_period('W').nunique()} weeks")
print(f"          x 5 hidden agents = {len(to_score) * 5:,} complete-a-line-up questions")

known_agents = {a for lineup in history["agents"] for a in lineup}
new_agents = sorted({a for lineup in to_score["agents"] for a in lineup} - known_agents)
maps_2025 = set(history.loc[history["played_at"].dt.year == 2025, "map"])
maps_not_in_2025 = sorted(set(to_score["map"]) - maps_2025)
print(f"\nagents never played before 2026: {new_agents or 'none'}")
print(f"maps not played in 2025: {maps_not_in_2025 or 'none'}")

## Run every method once

Gradient boosting is refitted every week and is slow to query, so this cell takes several
minutes.

In [ ]:
runs = {name: walk_forward(history, to_score, make) for name, make in METHODS.items()}

## The result

In [ ]:
results = pd.DataFrame({name: summarise(r) for name, r in runs.items()}).T
print(f"{len(to_score):,} line-ups from 2026.  Picking at random from the ~23 agents left gets top-3 about 13%.\n")
print(results.to_string(formatters={"top-1": "{:.1%}".format, "top-3": "{:.1%}".format,
                                    "from scratch, of 5": "{:.2f}".format}))

## The margin of error

In [ ]:
def per_lineup(result, cutoff):
    return (np.vstack(result["ranks"].to_numpy()) <= cutoff).mean(axis=1)

match_of = runs[HEADLINE]["match_id"].to_numpy()
match_ids = np.unique(match_of)
rows_of = {m: np.flatnonzero(match_of == m) for m in match_ids}
rng = np.random.default_rng(SEED)
draws = [np.concatenate([rows_of[m] for m in rng.choice(match_ids, len(match_ids))]) for _ in range(RESAMPLES)]

def interval(values):
    return np.percentile([values[d].mean() for d in draws], [2.5, 97.5])

for label, cutoff in [("top-3", 3), ("top-1", 1)]:
    v = per_lineup(runs[HEADLINE], cutoff)
    lo, hi = interval(v)
    print(f"{HEADLINE} {label}: {v.mean():.1%}   95% range {lo:.1%} to {hi:.1%}")

lead = per_lineup(runs[HEADLINE], 3) - per_lineup(runs[BAR], 3)
lead_lo, lead_hi = interval(lead)
print(f"\ntop-3 lead over the naive rule: {lead.mean() * 100:+.1f} points   "
      f"95% range {lead_lo * 100:+.1f} to {lead_hi * 100:+.1f}")

if lead.mean() > 0 and lead_lo > 0:
    verdict = "BEATS THE NAIVE RULE"
elif lead.mean() > 0:
    verdict = "AHEAD, BUT NOT DISTINGUISHABLE FROM THE NAIVE RULE"
else:
    verdict = "DOES NOT BEAT THE NAIVE RULE"
print(f"\nby the rule decided in advance: {verdict}")

## The two pre-registered breakdowns: what is new

In [ ]:
headline_run = runs[HEADLINE]

# 1. questions whose hidden agent had never been played before 2026
questions = pd.DataFrame(
    [(row, agent, rank)
     for row, rank_list in headline_run["ranks"].items()
     for agent, rank in zip(to_score.at[row, "agents"], rank_list)],
    columns=["row", "hidden agent", "rank"],
)
questions["new in 2026"] = questions["hidden agent"].isin(new_agents)
print("1. by whether the hidden agent was new in 2026")
for new, part in questions.groupby("new in 2026"):
    label = "new agent" if new else "agent seen before 2026"
    print(f"   {label:24} {len(part):5} questions   top-1 {(part['rank'] <= 1).mean():.1%}   "
          f"top-3 {(part['rank'] <= 3).mean():.1%}")
if new_agents:
    print("\n   the new agents, week by week (share of questions with the new agent in the top 3):")
    newq = questions[questions["new in 2026"]].assign(week=lambda d: headline_run.loc[d["row"], "week"].to_numpy())
    weekly = newq.groupby("week")["rank"].apply(lambda r: (r <= 3).mean())
    print("   " + "  ".join(f"{w:%d %b} {v:.0%}" for w, v in weekly.items()))

# 2. line-ups on maps not played in 2025
print("\n2. by whether the map was played in 2025")
on_absent = headline_run["map"].isin(maps_not_in_2025)
groups = [(f"{', '.join(maps_not_in_2025)} (not in 2025)", on_absent), ("maps played in 2025", ~on_absent)]
for label, mask in (groups if maps_not_in_2025 else [("every map was played in 2025", ~on_absent)]):
    part = headline_run[mask]
    print(f"   {label:28} {len(part):4} line-ups   " +
          "   ".join(f"{k} {v:.1%}" if k != "from scratch, of 5" else f"{k} {v:.2f}"
                     for k, v in summarise(part).items()))

## What this means

*Written after the cells above were run, in a separate commit. Everything above this line
was committed before running.*